<a href="https://colab.research.google.com/github/AmnaNoor123/urdu-ocr-codesaviours-si26-amna/blob/main/SI26_Week4_Amna_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**This notebook covers Week 4:**

Fine-tuning TrOCR on the Urdu OCR dataset.

Up till now we collected images (Week 1), preprocessed them (Week 2), and built the dataset class + train/test split (Week 3). This week we actually train a model on that data.

TrOCR is a transformer model from Microsoft — it has a vision encoder (reads the image) and a text decoder (outputs the characters). Instead of training from scratch, we load a version already trained on printed text and fine-tune it on our Urdu images. This is called **transfer learning**, and it's why we can get decent results with a small dataset instead of needing millions of images.

In this notebook, we will:

Reload the dataset from Week 3 (repo + labels + dataset class)
Load the pretrained TrOCR model
Set up the DataLoaders and optimiser
Train the model for a few epochs
Evaluate accuracy on the test set
Save the fine-tuned model to Google Drive


Reload Week 3 Setup — Repo, Labels, Dataset Class

In [7]:
!git clone https://github.com/AmnaNoor123/urdu-ocr-codesaviours-si26-amna.git
!cp urdu-ocr-codesaviours-si26-amna/labels.csv data/labels.csv


fatal: destination path 'urdu-ocr-codesaviours-si26-amna' already exists and is not an empty directory.
cp: cannot stat 'urdu-ocr-codesaviours-si26-amna/labels.csv': No such file or directory


In [8]:
!find urdu-ocr-codesaviours-si26-amna -maxdepth 2 -type f

urdu-ocr-codesaviours-si26-amna/SI26_Week2_amna.ipynb
urdu-ocr-codesaviours-si26-amna/SI26_Week3_Amna.ipynb
urdu-ocr-codesaviours-si26-amna/.git/index
urdu-ocr-codesaviours-si26-amna/.git/description
urdu-ocr-codesaviours-si26-amna/.git/packed-refs
urdu-ocr-codesaviours-si26-amna/.git/config
urdu-ocr-codesaviours-si26-amna/.git/HEAD
urdu-ocr-codesaviours-si26-amna/labels (1).csv
urdu-ocr-codesaviours-si26-amna/SI26_Week1_Amna.ipynb
urdu-ocr-codesaviours-si26-amna/README.md


In [9]:
!rm -rf urdu-ocr-codesaviours-si26-amna
!git clone https://github.com/AmnaNoor123/urdu-ocr-codesaviours-si26-amna.git
!find urdu-ocr-codesaviours-si26-amna -maxdepth 2 -type f

Cloning into 'urdu-ocr-codesaviours-si26-amna'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 68 (delta 29), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 57.27 KiB | 505.00 KiB/s, done.
Resolving deltas: 100% (29/29), done.
urdu-ocr-codesaviours-si26-amna/SI26_Week2_amna.ipynb
urdu-ocr-codesaviours-si26-amna/labels.csv
urdu-ocr-codesaviours-si26-amna/SI26_Week3_Amna.ipynb
urdu-ocr-codesaviours-si26-amna/.git/index
urdu-ocr-codesaviours-si26-amna/.git/description
urdu-ocr-codesaviours-si26-amna/.git/packed-refs
urdu-ocr-codesaviours-si26-amna/.git/config
urdu-ocr-codesaviours-si26-amna/.git/HEAD
urdu-ocr-codesaviours-si26-amna/SI26_Week1_Amna.ipynb
urdu-ocr-codesaviours-si26-amna/README.md


In [10]:
import os, shutil
os.makedirs('data', exist_ok=True)
shutil.copy('urdu-ocr-codesaviours-si26-amna/labels.csv', 'data/labels.csv')

import pandas as pd
df = pd.read_csv('data/labels.csv')
print('Total entries:', len(df))


Total entries: 200


In [11]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/Books.zip', 'r') as zip_ref:
    zip_ref.extractall('data/raw')
import os
print('Unzipped folders:', os.listdir('data/raw'))


Mounted at /content/drive
Unzipped folders: ['Other', 'Books', 'Sign boards', 'Handwritten', 'Newspaper', 'Synthetic raw images']


In [12]:
import os, shutil
import pandas as pd
import re
import unicodedata # Import unicodedata module

def clean_filename(name):
    # Ensure it's a string, strip leading/trailing whitespace first
    name = str(name).strip()

    # Convert to lowercase for case-insensitive matching (important for Linux filesystems)
    name = name.lower()

    # Normalize unicode characters to canonical composed form
    name = unicodedata.normalize('NFC', name)

    # Replace any non-standard whitespace characters with a single space
    # and strip again to handle potential internal leading/trailing spaces from this step
    name = re.sub(r'\s+', ' ', name).strip()

    base, ext = os.path.splitext(name)

    def decode_match(m):
        return chr(int(m.group(1), 16))

    # Handle #Uxxxx escapes in base name and extension
    base = re.sub(r'#U([0-9a-fA-F]{4,6})', decode_match, base)
    ext = re.sub(r'#U([0-9a-fA-F]{4,6})', decode_match, ext)

    # Remove trailing (number) like ' (1)', ' (2)'
    base = re.sub(r'\s*\(\d+\)$', '', base)

    full = base + ext

    # Remove duplicate extensions like '.png.png'
    full = re.sub(r'(\.\w+)\1$', r'\1', full)

    return full

# Assuming df is loaded from 'data/labels.csv'
# We need to process each entry in df['image'] and update the path.

# Step 1: Collect all actual image file paths on disk
# Maps cleaned_basename to a list of full paths where it was found
actual_image_files = {}
for root, _, files in os.walk('data/raw'):
    for fname in files:
        full_path_on_disk = os.path.join(root, fname)
        cleaned_fname_on_disk = clean_filename(fname)
        if cleaned_fname_on_disk not in actual_image_files:
            actual_image_files[cleaned_fname_on_disk] = []
        actual_image_files[cleaned_fname_on_disk].append(full_path_on_disk)

updated_image_paths = []
found_after_reconciliation_count = 0
not_found_final_count = 0
unresolved_paths_debug = [] # List to store paths that couldn't be resolved

print(f"Total unique cleaned filenames on disk: {len(actual_image_files)}")

for original_image_path_in_df in df['image']:
    dirname_from_df, basename_from_df = os.path.split(original_image_path_in_df)
    cleaned_basename_from_df = clean_filename(basename_from_df)

    resolved_path = None

    # Priority 1: Check for existence of the path derived from df with cleaned basename
    # This covers cases where original path was wrong but directory was right.
    candidate_path_from_df_cleaned_basename = os.path.join(dirname_from_df, cleaned_basename_from_df)
    if os.path.exists(candidate_path_from_df_cleaned_basename):
        resolved_path = candidate_path_from_df_cleaned_basename
    elif os.path.exists(original_image_path_in_df):
        # Fallback: if original_image_path_in_df itself exists (e.g., already clean)
        resolved_path = original_image_path_in_df

    if not resolved_path:
        # Priority 2: Deep search based on cleaned basename across all directories
        if cleaned_basename_from_df in actual_image_files:
            potential_paths_on_disk = actual_image_files[cleaned_basename_from_df]

            # Prioritize a path that's in the same directory as original_image_path_in_df
            # Compare cleaned directory names too for robustness
            cleaned_dirname_from_df = clean_filename(dirname_from_df) # Clean dirname from df entry
            matching_in_original_dir = []
            for p_disk in potential_paths_on_disk:
                disk_dirname_cleaned = clean_filename(os.path.split(p_disk)[0])
                if disk_dirname_cleaned == cleaned_dirname_from_df:
                    matching_in_original_dir.append(p_disk)

            if len(matching_in_original_dir) == 1:
                resolved_path = matching_in_original_dir[0]
                found_after_reconciliation_count += 1
            elif len(potential_paths_on_disk) == 1:
                # If unique overall, use that one (even if directory doesn't match clean_dirname_from_df)
                resolved_path = potential_paths_on_disk[0]
                found_after_reconciliation_count += 1
            elif len(potential_paths_on_disk) > 1:
                # If multiple ambiguous matches and no clear directory match, pick the first one.
                # This is a heuristic to ensure we find *an* image if multiple exist with same basename.
                resolved_path = potential_paths_on_disk[0]
                found_after_reconciliation_count += 1

    if resolved_path:
        updated_image_paths.append(resolved_path)
    else:
        not_found_final_count += 1
        updated_image_paths.append(original_image_path_in_df) # Keep original for debug, it will be filtered
        unresolved_paths_debug.append(original_image_path_in_df) # Store for debugging
        # Print debug info for each unresolved path
        print(f"DEBUG: Unresolved: {original_image_path_in_df}")
        print(f"  Cleaned basename from df: '{cleaned_basename_from_df}'")
        print(f"  Available on disk for this cleaned basename: {actual_image_files.get(cleaned_basename_from_df, 'None')}")


# Update the 'image' column in the DataFrame
df['image'] = updated_image_paths

# Save the modified DataFrame back to 'labels.csv'
df.to_csv('data/labels.csv', index=False)

print('labels.csv updated with reconciled image paths.')
print(f'Reconciled {found_after_reconciliation_count} entries via deep search and cleaning.')
if not_found_final_count > 0:
    print(f'Warning: {not_found_final_count} image paths from labels.csv were not found on disk.')
    print('These entries will be filtered by the dataset loader, resulting in fewer than 200 samples.')
    print('Unresolved paths (first 5):', unresolved_paths_debug[:5])
else:
    print('All 200 image paths from labels.csv were successfully reconciled with files on disk.')

Total unique cleaned filenames on disk: 200
labels.csv updated with reconciled image paths.
Reconciled 20 entries via deep search and cleaning.
All 200 image paths from labels.csv were successfully reconciled with files on disk.


Dataset Class (same as Week 3)

In [13]:
!pip install transformers torch pillow pandas sentencepiece

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import os
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor

        # Re-introducing filtering logic to prevent FileNotFoundError
        initial_count = len(self.data)
        self.data['image_exists'] = self.data['image'].apply(lambda x: os.path.exists(x))
        self.data = self.data[self.data['image_exists']].drop(columns=['image_exists'])
        filtered_count = len(self.data)

        if initial_count > filtered_count:
            print(f'Warning: Removed {initial_count - filtered_count} entries due to missing image files.')
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)

        return {'pixel_values': pixel_values, 'labels': labels}

In [14]:
from transformers import TrOCRProcessor, AutoImageProcessor, RobertaTokenizer

# Instantiate image processor
image_processor = AutoImageProcessor.from_pretrained('microsoft/trocr-base-printed', backend="pil")

# Instantiate tokenizer with use_fast=False
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed', use_fast=False)

# Combine them into a processor
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

dataset = UrduOCRDataset('data/labels.csv', processor)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 40


## Step 1 — Load the Pretrained TrOCR Model


In [15]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoImageProcessor, RobertaTokenizer
import torch

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

# Load the pretrained model components
image_processor = AutoImageProcessor.from_pretrained('microsoft/trocr-base-printed', backend="pil")
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed', use_fast=False)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

# Configure model for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda


config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


## Step 2 — Set Up Training

A `DataLoader` wraps the dataset and feeds it to the model in small batches — batch size 4 means the model sees 4 images at a time before updating its weights. `AdamW` is the optimiser that controls how the model adjusts itself after each batch.

In [16]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

# Optimiser
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 40
Ready to train!


In [17]:
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    processed_batches = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        try:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            processed_batches += 1 # Increment only for successful batches
            if batch_idx % 10 == 0:
                print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
        except FileNotFoundError as e:
            print(f"  Skipping batch {batch_idx} due to FileNotFoundError: {e}")
            continue # Skip to the next batch

    if processed_batches > 0:
        avg_loss = total_loss / processed_batches # Use processed_batches for accurate average
        print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')
    else:
        print(f'Epoch {epoch + 1} complete | No batches processed successfully.')

print('\nTraining complete!')


Epoch 1/3
------------------------------
  Batch 0/40 | Loss: 17.8052
  Batch 10/40 | Loss: 3.1862
  Batch 20/40 | Loss: 1.6889
  Batch 30/40 | Loss: 1.7344
Epoch 1 complete | Average Loss: 2.5334

Epoch 2/3
------------------------------
  Batch 0/40 | Loss: 0.5807
  Batch 10/40 | Loss: 1.2954
  Batch 20/40 | Loss: 1.7064
  Batch 30/40 | Loss: 1.2464
Epoch 2 complete | Average Loss: 1.3004

Epoch 3/3
------------------------------
  Batch 0/40 | Loss: 2.2629
  Batch 10/40 | Loss: 2.2233
  Batch 20/40 | Loss: 1.2192
  Batch 30/40 | Loss: 1.7534
Epoch 3 complete | Average Loss: 1.0980

Training complete!


## Step 3 — Evaluate Your Model

Now we test the model on images it has never seen — the test split from Week 3. `model.eval()` turns off weight updates so evaluation doesn't affect the trained model.

In [18]:
model.eval()
print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )
        actual_text = processor.batch_decode(
            labels, skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual:    {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')


=== Model Evaluation on Test Images ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted: ���������ڌ��������
Actual:    ٹماٹر، پیاز

Predicted: �
Actual:    ب

Predicted: ��ڌ���������������
Actual:    میں کھانا گرم کرکے آتی ہوں

Predicted: ���
Actual:    ڑ

Predicted: ���������ڌ��������
Actual:    وہاں ریاض مسلسل سے کام چلتا ہے، یہاں گلے کے سہارے کلام چلتا ہے

Predicted: �������������������
Actual:    پاکستان کے شمالی علاقے بہت خوبصورت ہیں

Predicted: ���ک��������������
Actual:    آج کا موسم خوشگوار ہے

Predicted: ���ڌ��������������
Actual:    میں گانا گانا جانتی ہوں

Predicted: ��ک��� ک��� � � �
Actual:    درخت، مشکل، یونٹ، جب تک، معمول

Predicted: ��
Actual:    ز

Predicted: �
Actual:    ر

Predicted: �������������������
Actual:    ماں نے کھانا بنایا

Predicted: �����ک������������
Actual:    راز، تم مرد ہو کر عورت کو بدنام کرتے ہو

Predicted: ��
Actual:    ع

Predicted: ��
Actual:    چ

Predicted: ��ک������������ڌ�
Actual:    میں سالار سکندر سے بات کرنا چاہتی ہوں، آپ امامہ ہاشم ہیں؟

Predicted: ��
Actual:    آ

Predicted: �
Actual:    د

Predicted: �ک����������


**Discussion Point : TrOCR Tokenizer Limitation on Urdu Script**

**Final Result:** Accuracy: 0.0% (0/40 correct) on the test set after 3 epochs of fine-tuning, despite training loss dropping steadily (17.8 → 2.53 → 1.30 → 1.10 average loss across epochs).

**What went wrong:** Model predictions consistently decoded as `�` (replacement characters) instead of valid Urdu text, even though the loss curve suggested the model was learning.

**Why this happened:** `microsoft/trocr-base-printed` uses a RobertaTokenizer, which is a byte-level tokenizer trained on English/Latin text. Since Arabic-script characters (used for Urdu) weren't part of its original vocabulary, each Urdu character has to be reconstructed from a precise sequence of byte-level tokens — if even one byte in that sequence is predicted incorrectly, the whole character fails to decode and shows as `�`. This explains the disconnect: loss is computed per-token, so partial correctness lowers loss, but exact character/sentence match (used for accuracy) requires every byte in the sequence to be correct.

**Example mismatches:**
- Actual: `ٹماٹر، پیاز` → Predicted: `���������ڌ��������`
- Actual: `ب` (single letter) → Predicted: `�`
- Actual: `میں کھانا گرم کرکے آتی ہوں` → Predicted: `��ڌ���������������`

**Takeaway:** A pretrained English OCR model's tokenizer is a hard bottleneck for Urdu — fine-tuning on more data or more epochs alone won't fix this, since the underlying vocabulary can't represent Urdu characters efficiently. A model with a multilingual or Urdu-aware tokenizer (e.g., a mBERT/XLM-R-based decoder, or an Urdu-specific OCR checkpoint) would be a more realistic direction for Week 5+.


## Step 4 — Save  Model


In [19]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'Model saved to Google Drive: {save_path}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
